# 01 Query Examples

Inspect generated PubMed queries and LLM prompt templates before running data collection.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.screening import UNIFIED_AMR_INTEREST_SYSTEM_PROMPT
from amr.pipelines import (
    print_pubmed_query_examples,
    print_unified_prompt_examples,
    print_wrapped_system_prompt,
)

## Glossary

- **AMR**: Antimicrobial Resistance.
- **SI**: Scientific Interest score, defined as residual AMR attention after background adjustment.
- **TIAB**: Title/Abstract PubMed field selector (search terms constrained to title or abstract text).
- **PMID**: PubMed Identifier for a publication record.
- **LLM**: Large Language Model used for abstract screening.
- **BG / background counts**: Yearly publication marginals for pathogens or antibiotics used in SI baseline modeling.
- **Slice**: A chosen year interval (`SLICE_START_YEAR` to `SLICE_END_YEAR`) processed in one run.


## Configuration

Keep these parameters aligned with `01_collect_yearly_data.ipynb` before running collection.

In [3]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PARQUET_OUTPUT_DIR = OUTPUT_DIR / "parquets"
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR

PATHOGENS_PATH = ASSETS_DIR / "bacteria_synonym_lookup.csv"
PATHOGEN_PRIORITY_PATH = ASSETS_DIR / "counts_extended2.csv"
MIN_PATHOGENIC_INDEX = 0.0
ANTIBIOTIC_COARSENING_PATH = ASSETS_DIR / "antibiotic_coarsening_lookup.csv"
ANTIBIOTIC_SYNONYM_PATH = ASSETS_DIR / "antibiotic_synonym_lookup.csv"
OUTPUT_YEARLY_AMR_PATH = DATA_DIR / "candidate_amr_pair_year_counts.parquet"
OUTPUT_YEARLY_SCREEN_PATH = DATA_DIR / "yearly_amr_screened_pairs.parquet"
OUTPUT_UNIFIED_DECISIONS_PATH = DATA_DIR / "yearly_amr_unified_decisions.parquet"
OUTPUT_PATHOGEN_BG_PATH = DATA_DIR / "pathogen_tiab_counts.parquet"
OUTPUT_ANTIBIOTIC_BG_PATH = DATA_DIR / "antibiotic_tiab_counts.parquet"
INTERVAL_PMIDS_PATH = DATA_DIR / "pmids_results.parquet"
YEARLY_PAIR_PMID_PATH = DATA_DIR / "candidate_amr_pair_pmid_links.parquet"
PUBMED_METADATA_CACHE_PATH = DATA_DIR / "pubmed_pmid_metadata.parquet"
ABSTRACT_CACHE_PATH = DATA_DIR / "pmid_abstracts.parquet"
PUBMED_EMAIL = "xfcosta@gmail.com"  # replace with a real contact email

# Auto-resume behavior: all collection/filtering stages reuse on-disk outputs
# and continue only the missing work.

# Partition-friendly execution window: run a slice now, merge into canonical stores.
TOTAL_START_YEAR = 1970
TOTAL_END_YEAR = 2026
TOTAL_MIN_DATE = f"{TOTAL_START_YEAR}/01/01"
TOTAL_MAX_DATE = f"{TOTAL_END_YEAR}/12/31"
SLICE_START_YEAR = 2000
SLICE_END_YEAR = 2026
MIN_DATE = f"{SLICE_START_YEAR}/01/01"
MAX_DATE = f"{SLICE_END_YEAR}/12/31"
INTERVAL_YEAR_GAP = 5
NUM_PATHOGENS = 5  # use None for all pathogens; increase over time to expand cumulatively
NUM_ANTIBIOTICS = 10  # use None for all antibiotics; cache/resume skips previously completed work
# Existing notebook intent: AMR-focused retrieval before unified LLM relevance screen.
EXTRA_AMR_KEYWORDS = [
    "\"antimicrobial resistance\"[Title/Abstract] OR \"antibiotic resistance\"[Title/Abstract]"
]
RUN_LLM_FILTER = True
MAX_LLM_CALLS = None
# Editable per-run prompt (single source of truth for this notebook execution)
UNIFIED_SYSTEM_PROMPT = UNIFIED_AMR_INTEREST_SYSTEM_PROMPT
# Throughput/reliability controls for unified LLM filter
LLM_MAX_WORKERS = 2
LLM_BATCH_SIZE = 8
FAILED_RETRY_PATH = PARQUET_OUTPUT_DIR / "yearly_amr_unified_failures.parquet"
MAX_FAILURE_RETRIES = 5
ABSTRACT_FETCH_BATCH_SIZE = 200
PREFETCH_ABSTRACTS = True
# Batch-style collection logging controls
COLLECTION_VERBOSE = 1  # 0=silent, 1=batch summaries, 2=detailed per interval
COLLECTION_PROGRESS_EVERY = 300
PAIR_FULL_FETCH_THRESHOLD = 9500  # fetch once and partition locally below this PubMed count
BACKGROUND_SCOPE = "observed_entities"  # "observed_entities" is fast; "all_entities" is canonical/full
BACKGROUND_FETCH_THRESHOLD = 9500
BACKGROUND_METADATA_BATCH_SIZE = 200
LLM_CATEGORY_BACKFILL_DECISIONS = ("yes", "uncertain")  # prioritize SI-contributing rows; use None for all rows
# Unified filter logging cadence
UNIFIED_PROGRESS_EVERY = 200



## Load Prepared Pathogen And Antibiotic Inputs

In [4]:
bacteria_synonym_df = pd.read_csv(PATHOGENS_PATH)
antibiotic_df = pd.read_csv(ANTIBIOTIC_COARSENING_PATH)
antibiotic_synonym_df = pd.read_csv(ANTIBIOTIC_SYNONYM_PATH)

if not PATHOGEN_PRIORITY_PATH.exists():
    raise FileNotFoundError(
        f"Pathogen priority file not found: {PATHOGEN_PRIORITY_PATH}. "
        "Run notebooks/00_prepare_inputs.ipynb first."
    )

pathogen_priority_df = pd.read_csv(PATHOGEN_PRIORITY_PATH)
required_priority_cols = {"CanonicalSpecies", "pathogenic_index"}
missing_priority_cols = required_priority_cols - set(pathogen_priority_df.columns)
if missing_priority_cols:
    raise KeyError(
        f"{PATHOGEN_PRIORITY_PATH} is missing required columns: {sorted(missing_priority_cols)}"
    )

pathogen_priority_df = pathogen_priority_df.copy()
pathogen_priority_df["CanonicalSpecies"] = pathogen_priority_df["CanonicalSpecies"].astype(str).str.strip()
pathogen_priority_df["pathogenic_index"] = pd.to_numeric(
    pathogen_priority_df["pathogenic_index"],
    errors="coerce",
)
pathogen_priority_df = pathogen_priority_df.dropna(subset=["CanonicalSpecies", "pathogenic_index"])
pathogen_priority_df = pathogen_priority_df.loc[
    pathogen_priority_df["pathogenic_index"] > float(MIN_PATHOGENIC_INDEX)
]
pathogen_priority_df = pathogen_priority_df.sort_values(
    ["pathogenic_index", "CanonicalSpecies"],
    ascending=[False, True],
).drop_duplicates(subset=["CanonicalSpecies"], keep="first")

all_pathogens = pathogen_priority_df["CanonicalSpecies"].tolist()
all_antibiotics = sorted(antibiotic_df["Antibiotic"].dropna().astype(str).str.strip().unique())

pathogens = all_pathogens if NUM_PATHOGENS is None else all_pathogens[: int(NUM_PATHOGENS)]
antibiotics = all_antibiotics if NUM_ANTIBIOTICS is None else all_antibiotics[: int(NUM_ANTIBIOTICS)]

print(f"Pathogen priority source: {PATHOGEN_PRIORITY_PATH}")
print(f"Full synonym lookup vocabulary: {bacteria_synonym_df['bacteria name'].nunique():,}")
print(
    f"Pathogenic-index pathogen vocabulary: {len(all_pathogens):,} "
    f"(MIN_PATHOGENIC_INDEX>{MIN_PATHOGENIC_INDEX})"
)
print(f"Antibiotic coarsening source: {ANTIBIOTIC_COARSENING_PATH}")
print(f"Antibiotic synonym source: {ANTIBIOTIC_SYNONYM_PATH} ({len(antibiotic_synonym_df):,} rows)")
print(f"Full antibiotic vocabulary: {len(all_antibiotics):,}")
print(f"Active pathogens this run: {len(pathogens):,} (NUM_PATHOGENS={NUM_PATHOGENS})")
print(f"Active antibiotics this run: {len(antibiotics):,} (NUM_ANTIBIOTICS={NUM_ANTIBIOTICS})")
print("Top active pathogens by pathogenic_index:")
display(
    pathogen_priority_df[["CanonicalSpecies", "pathogenic_index", "n_causation", "n_articles"]]
    .head(len(pathogens))
    .reset_index(drop=True)
)



Pathogen priority source: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/counts_extended2.csv
Full synonym lookup vocabulary: 19,956
Pathogenic-index pathogen vocabulary: 915 (MIN_PATHOGENIC_INDEX>0.0)
Antibiotic coarsening source: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/antibiotic_coarsening_lookup.csv
Antibiotic synonym source: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/antibiotic_synonym_lookup.csv (422 rows)
Full antibiotic vocabulary: 245
Active pathogens this run: 5 (NUM_PATHOGENS=5)
Active antibiotics this run: 10 (NUM_ANTIBIOTICS=10)
Top active pathogens by pathogenic_index:


,CanonicalSpecies,pathogenic_index,n_causation,n_articles
0,Salmonella enterica,1.000000,554,24694
1,Mycobacterium tuberculosis,0.928144,299,20559
2,Clostridioides difficile,0.910833,298,17012
3,Haemophilus influenzae,0.860892,200,14537
4,Klebsiella pneumoniae,0.857160,176,15847


## PubMed Query Logic for Abstract Collection

The collection query is assembled per `(pathogen, antibiotic)` pair with four components:

1. **Antibiotic clause**: antibiotic name in `Abstract`, including split variants for hyphen/slash names (for example `co-trimoxazole` as exact phrase OR split terms).
2. **Pathogen clause**: pathogen + known synonyms in `Abstract` (`OR`-joined).
3. **AMR clause**: additional AMR-focused condition from `EXTRA_AMR_KEYWORDS` in `Title/Abstract`.
4. **Date clause**: publication date restricted to `[MIN_DATE, MAX_DATE]`.

Final query shape:

`(antibiotic_clause AND pathogen_clause) AND extra_amr_clause AND date_range_clause`


In [5]:
print_pubmed_query_examples(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    antibiotic_synonym_df=antibiotic_synonym_df,
    extra_clauses=EXTRA_AMR_KEYWORDS,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    max_examples=3,
)

Sample pathogen: Salmonella enterica
Sample antibiotic: Amikacin

Sample PubMed query:

(("Amikacin"[Abstract]) AND ("Salmonella enterica"[Abstract] OR "salmonella arizonae"[Abstract] OR "salmonella choleraesuis"[Abstract] OR "salmonella enterica"[Abstract] OR "salmonella enteritidis"[Abstract] OR "salmonella paratyphi"[Abstract] OR "salmonella typhi"[Abstract] OR "salmonella typhimurium"[Abstract])) AND ("antimicrobial resistance"[Title/Abstract] OR "antibiotic resistance"[Title/Abstract]) AND ("2000/01/01"[Date - Publication] : "2026/12/31"[Date - Publication])


EXAMPLE PUBMED QUERIES (MULTIPLE VIEWS)

Example 1: Salmonella enterica vs Amikacin
- Pathogen TIAB clause:
("Salmonella enterica"[Title/Abstract] OR "salmonella arizonae"[Title/Abstract] OR "salmonella choleraesuis"[Title/Abstract] OR "salmonella enterica"[Title/Abstract] OR "salmonella enteritidis"[Title/Abstract] OR "salmonella paratyphi"[Title/Abstract] OR "salmonella typhi"[Title/Abstract] OR "salmonella typhimurium"[Ti

## Unified LLM Prompt Templates
This workflow uses a single strict LLM screen:
- **System prompt**: policy for AMR evidence + scientific-interest relevance.
- **User prompt template**: PMID-specific payload (pathogen, antibiotic, title, abstract).
**Prompt governance note:** if you change this prompt between slices, document it so downstream results remain interpretable.


In [6]:
print_wrapped_system_prompt(UNIFIED_SYSTEM_PROMPT, width=80)
print("\n" + "=" * 80 + "\n")
print_unified_prompt_examples(UNIFIED_SYSTEM_PROMPT)

PRETTY PRINT (80 COLS) OF UNIFIED_SYSTEM_PROMPT:

You are classifying scientific abstracts about antimicrobial resistance (AMR).

Task:
Given a target pathogen, a target antibiotic or antibiotic class, a title, and
an abstract,
classify the abstract into exactly one primary category describing why the paper
contributes
to scientific interest for the target pathogen-antibiotic pair.

Core rule:
Do not classify by keyword co-occurrence. First determine whether the abstract
clearly links
the target pathogen to the target antibiotic/class. Only assign a substantive
category if the
target pair relation is explicitly supported by the title or abstract.

If the target pathogen-antibiotic relation is unclear, ambiguous, incidental, or
unsupported,
discard it by setting:
- decision = "no"
- primary_category = "NONE"
- target_pair_relation_supported = false

Attribution rules:
A pathogen is linked to resistance, susceptibility, testing, treatment, or AMR
relevance for an
antimicrobial agent only